### échanger avec un serveur distant avec la mméthode REST et le protocole HTTP

In [ ]:
# ------------------ utilisation naïve de request pour chercher cdes données -----------
"""
chercher une liste d'objets utilisateurs sur le domaine gorest.co.in
avec l'endpoint /public/v2/users avec la méthode GET
"""

import requests

url = "https://gorest.co.in/public/v2/users"

try:
    r = requests.get(url)
    if 200 <= r.status_code < 300 and "application/json" in r.headers["content-type"]:
        data = r.json()
    else:
        raise ValueError(f"pb: status: {r.status_code}, headers: {r.headers["content-type"]}")
except (requests.ConnectionError, requests.HTTPError, ValueError) as e:
    print(e)

data

In [ ]:
# ---------------------- fonctions hasattr, setattr, getattr ------------------
class Truc:
    pass

t = Truc()

# vérifier si on a un attribut "param" dans t
hasattr(t, "param")
# créer un attribut nommé "param" dynamiquement  dans t
setattr(t, "param", 10)
# retourner dynamiquement l'attribut "param" dans t
getattr(t, "param")



In [ ]:
# ---------------------------- client en poo pour intéragir avec gorest.co.in -------------

"""
1/ créer une classe qui a comme attribut FIXES
  - le domain: gorest.co.in
  - la version: v2
  - radical: /public

2/ créer une méthode publique get_user avec un entier comme paramètre (id)
  - qui retourne un dictionnaire qui a 
    + une clé "data" ==> l'objet json de l'utilisateur
    + error ==> si erreur
  - cette méthode va déléguer l'utilisation de requests à une méthode privée centrale 
    __call(method, endpoint)

3/ créer la méthode privée __call(method, endpoint)
  - qui créer l'url à partir de ses éléments
  - utiliser intelligemment la cellule précédente pour faire varier la méthode de requests dynamiquement
  - gère la réponse côté data et error
"""
import requests
import os
# pip install python-dotenv
from dotenv import load_dotenv

# charge les variables contenues dans le fichier .env en variable d'environnement pour le processus (!= permanente)
load_dotenv()

TOKEN = os.environ.get("GOREST_TOKEN")

class GoRestV2Client:
    def __init__(self):
        self.domain = "gorest.co.in"
        self.radical = "public"
        self.version = "v2"
        self.token = TOKEN

    def __call(self, method: str, endpoint: str, data: dict={}, headers: dict={}) -> dict:
        url = f"https://{self.domain}/{self.radical}/{self.version}/{endpoint}"
        ret = {"data": None, "error": None}
        if method.upper() in ("POST", "PUT", "PATCH", "DELETE"):
            headers["Authorization"] = f"Bearer {self.token}"

        try:
            call_function = getattr(requests, method.lower())
            r = call_function(url, data=data, headers=headers)
            if 200 <= r.status_code < 300 and "application/json" in r.headers["content-type"]:
                ret["data"] = r.json()
            else:
                raise ValueError(f"pb: status: {r.status_code}, headers: {r.headers["content-type"]}")
        except (requests.ConnectionError, requests.HTTPError, ValueError) as e:
            ret["error"] = e
        return ret

    def get_user(self, _id: int) -> dict:
        ret = self.__call("GET", f"users/{_id}")
        return ret

    def post_user(self, user: dict) -> dict:
        ret = self.__call("POST", f"users", data=user)
        return ret


api = GoRestV2Client()
# print(api.get_user(8690763))
user = {
    "name": "mott LAMAM",
    "email": "me@example.com",
    "gender": "male",
    "status": "active"
}
print(api.post_user(user))
